# 05 — RAG: Retrieval-Augmented Generation

**Tickets:** R-01, R-02, R-03, R-04, R-05, R-06  
**Business Question (BQ-5):** Can a conversational interface answer ad-hoc questions about NYC taxi operations & policy?  
**Purpose:** Prepare a document corpus, generate embeddings, build a retrieval function, and wire up an LLM to answer natural-language questions.

---

## Setup

In [0]:
# Setup — imports, config, embedding client
from mlflow.deployments import get_deploy_client
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    DoubleType,
    StringType,
    StructField,
    StructType,
)

CATALOG = "students_data"
SCHEMA = "`ethan-hawthorne`"
EMBEDDING_ENDPOINT = "databricks-gte-large-en"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

deploy_client = get_deploy_client("databricks")

print(f"✓ Setup complete — catalog={CATALOG}, schema={SCHEMA}")
print(f"  Embedding endpoint: {EMBEDDING_ENDPOINT}")

## R-01 & R-02 — Corpus preparation & chunking

In [0]:
# R-01 — Collect & prepare RAG corpus documents
corpus_documents = [
    {
        "doc_id": "tlc_rules",
        "source": "NYC Taxi & Limousine Commission",
        "title": "TLC Rules & Regulations",
        "content": (
            "All NYC yellow taxis must operate with a valid TLC medallion. "
            "Drivers must hold a valid TLC driver license. "
            "The taximeter must be used for all trips and must be engaged at the start of each ride. "
            "Drivers must accept any passenger requesting a trip to any destination within the five boroughs of New York City. "
            "The passenger has the right to choose the method of payment (cash or card). "
            "Air conditioning is required from May 1 through October 31 when the outside temperature exceeds 70°F. "
            "Drivers must offer a printed receipt at the end of every trip. "
            "Drivers may not refuse passengers based on their destination."
        ),
    },
    {
        "doc_id": "taxi_faqs",
        "source": "NYC Taxi & Limousine Commission",
        "title": "NYC Taxi Frequently Asked Questions",
        "content": (
            "Yellow cabs can be street-hailed in all five boroughs of New York City. "
            "Green Boro Taxis serve the outer boroughs and upper Manhattan above 110th Street. "
            "Accepted payment methods include cash, credit or debit card, and contactless or mobile payments. "
            "Tipping is not required but is customary at 15–20% of the fare. "
            "For lost property in a taxi, call 311. "
            "To file a complaint about a driver or trip, contact the TLC by calling 311 or visiting nyc.gov/tlc. "
            "Wheelchair-accessible taxis are available across the city."
        ),
    },
    {
        "doc_id": "pricing_policy",
        "source": "NYC Taxi & Limousine Commission",
        "title": "NYC Taxi Pricing Policy",
        "content": (
            "The initial metered fare charge is $3.00. "
            "The meter rate is $0.70 per 1/5 mile traveled. "
            "The meter also charges $0.70 per 60 seconds when the taxi is in slow traffic or idle. "
            "A rush hour surcharge of $2.50 applies Monday through Friday from 4:00 PM to 8:00 PM. "
            "An overnight surcharge of $1.00 applies from 8:00 PM to 6:00 AM. "
            "An MTA State Surcharge of $0.50 is added to every ride. "
            "An improvement surcharge of $1.00 is added to every ride. "
            "A congestion surcharge of $2.50 applies to all trips that begin, end, or pass through Manhattan below 96th Street. "
            "The flat fare from JFK Airport to Manhattan is $70.00, plus tolls and tip. "
            "Trips to Newark Airport are charged at the metered fare plus a $20.00 surcharge. "
            "All tolls incurred during the trip are the responsibility of the passenger."
        ),
    },
]

print(f"✓ R-01 complete: {len(corpus_documents)} corpus documents prepared")
for doc in corpus_documents:
    print(f"  - {doc['doc_id']}: {doc['title']} ({len(doc['content'])} chars)")

In [0]:
# R-02 — Chunk documents into retrieval-friendly segments


def chunk_document(doc, max_chars=1600, overlap_chars=200):
    """Split a document into chunks on sentence boundaries with overlap.

    Target: 300-500 tokens (~1200-2000 chars at ~4 chars/token).
    Default max_chars=1600 (~400 tokens), overlap=200 (~50 tokens).
    """
    text = doc["content"]
    sentences = [s.strip() + "." for s in text.split(".") if s.strip()]

    chunks = []
    current_chunk = ""
    chunk_idx = 0

    for sentence in sentences:
        if len(current_chunk) + len(sentence) > max_chars and current_chunk:
            chunks.append(
                {
                    "chunk_id": f"{doc['doc_id']}_chunk_{chunk_idx}",
                    "doc_id": doc["doc_id"],
                    "source": doc["source"],
                    "title": doc["title"],
                    "content": current_chunk.strip(),
                }
            )
            # Keep overlap from end of current chunk
            overlap_text = (
                current_chunk[-overlap_chars:]
                if len(current_chunk) > overlap_chars
                else current_chunk
            )
            current_chunk = overlap_text + " " + sentence
            chunk_idx += 1
        else:
            current_chunk = (current_chunk + " " + sentence).strip()

    if current_chunk.strip():
        chunks.append(
            {
                "chunk_id": f"{doc['doc_id']}_chunk_{chunk_idx}",
                "doc_id": doc["doc_id"],
                "source": doc["source"],
                "title": doc["title"],
                "content": current_chunk.strip(),
            }
        )

    return chunks


# Chunk all corpus documents
corpus_chunks = []
for doc in corpus_documents:
    chunks = chunk_document(doc)
    corpus_chunks.extend(chunks)

print(
    f"\u2713 R-02 complete: {len(corpus_chunks)} chunks"
    f" from {len(corpus_documents)} documents"
)
for chunk in corpus_chunks:
    print(
        f"  - {chunk['chunk_id']}: {len(chunk['content'])} chars"
        f" (~{len(chunk['content']) // 4} tokens)"
    )

## R-03 — Generate embeddings & store as vector table

In [0]:
# R-03 — Generate embeddings for each chunk & store as Delta table

TABLE_NAME = "rag_corpus_embeddings"


def _embed_texts(texts: list[str]) -> list[list[float]]:
    """Call the embedding endpoint for a batch of texts."""
    response = deploy_client.predict(
        endpoint=EMBEDDING_ENDPOINT,
        inputs={"input": texts},
    )
    return [item["embedding"] for item in response.data]


# Generate embeddings for every chunk
contents = [chunk["content"] for chunk in corpus_chunks]
embeddings = _embed_texts(contents)

print(f"  Generated {len(embeddings)} embeddings")
print(f"  Dimensions per vector: {len(embeddings[0])}")

# Build rows: chunk metadata + embedding vector
rows = []
for chunk, emb in zip(corpus_chunks, embeddings):
    rows.append(
        {
            "chunk_id": chunk["chunk_id"],
            "doc_id": chunk["doc_id"],
            "source": chunk["source"],
            "title": chunk["title"],
            "content": chunk["content"],
            "embedding": emb,
        }
    )

schema = StructType(
    [
        StructField("chunk_id", StringType(), False),
        StructField("doc_id", StringType(), False),
        StructField("source", StringType(), False),
        StructField("title", StringType(), False),
        StructField("content", StringType(), False),
        StructField("embedding", ArrayType(DoubleType()), False),
    ]
)

df_embeddings = spark.createDataFrame(rows, schema=schema)

fqn = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"
df_embeddings.write.format("delta").mode("overwrite").saveAsTable(fqn)

print(f"\n\u2713 R-03 complete: saved {df_embeddings.count()} rows to {fqn}")
display(
    df_embeddings.select(
        "chunk_id", "doc_id", "title", F.size("embedding").alias("vector_dim")
    )
)

## R-04 — Retrieval function

In [0]:
# R-04 — Retrieval function: embed query, cosine-similarity search, return top-k
import numpy as np


def _cosine_similarity(vec_a, vec_b):
    """Compute cosine similarity between two vectors."""
    a = np.asarray(vec_a)
    b = np.asarray(vec_b)
    dot = float(np.dot(a, b))
    norm = float(np.linalg.norm(a) * np.linalg.norm(b))
    return dot / norm if norm != 0 else 0.0


def retrieve(query: str, top_k: int = 3) -> list[dict]:
    """Embed *query*, score against the vector table, return top-k chunks.

    Returns a list of dicts with keys:
        chunk_id, doc_id, title, content, score
    """
    # 1. Embed the query
    response = deploy_client.predict(
        endpoint=EMBEDDING_ENDPOINT,
        inputs={"input": [query]},
    )
    query_vec = response.data[0]["embedding"]

    # 2. Read the vector table to the driver (small corpus — safe to collect)
    fqn = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"
    rows = (
        spark.table(fqn)
        .select("chunk_id", "doc_id", "title", "content", "embedding")
        .collect()
    )

    # 3. Score every chunk on the driver
    scored = []
    for row in rows:
        score = _cosine_similarity(query_vec, row["embedding"])
        scored.append(
            {
                "chunk_id": row["chunk_id"],
                "doc_id": row["doc_id"],
                "title": row["title"],
                "content": row["content"],
                "score": round(score, 6),
            }
        )

    # 4. Rank and return top-k
    scored.sort(key=lambda r: r["score"], reverse=True)
    return scored[:top_k]


# --- Quick validation ---
test_queries = [
    "What is the flat fare from JFK to Manhattan?",
    "Can I pay by credit card in a yellow cab?",
    "What surcharges apply during rush hour?",
]

for q in test_queries:
    print(f"\n\U0001f50d Query: {q}")
    hits = retrieve(q, top_k=2)
    for i, hit in enumerate(hits, 1):
        print(f"  {i}. [{hit['score']:.4f}] {hit['title']}")
        print(f"     {hit['content'][:120]}...")

print("\n\u2713 R-04 complete: retrieve() validated")

## R-05 — RAG pipeline (prompt + retrieval + LLM)

In [0]:
# R-05 — RAG pipeline: prompt template + retrieval + LLM answer generation

LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

RAG_PROMPT_TEMPLATE = """You are a helpful NYC taxi operations assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.
Keep your answer concise (3-5 sentences) and cite which source document
supported each claim.

--- CONTEXT ---
{context}
--- END CONTEXT ---

Question: {question}

Answer:"""


def rag_answer(question: str, top_k: int = 2) -> dict:
    """End-to-end RAG: retrieve chunks, build prompt, call LLM.

    Returns a dict with keys: question, answer, sources.
    """
    # 1. Retrieve relevant chunks
    hits = retrieve(question, top_k=top_k)

    # 2. Build context block from retrieved chunks
    context_parts = []
    for i, hit in enumerate(hits, 1):
        context_parts.append(f"[Source {i}: {hit['title']}]\n{hit['content']}")
    context = "\n\n".join(context_parts)

    # 3. Fill prompt template
    prompt = RAG_PROMPT_TEMPLATE.format(
        context=context,
        question=question,
    )

    # 4. Call LLM
    llm_response = deploy_client.predict(
        endpoint=LLM_ENDPOINT,
        inputs={
            "messages": [
                {"role": "user", "content": prompt},
            ],
            "max_tokens": 512,
            "temperature": 0.1,
        },
    )
    answer = llm_response.choices[0]["message"]["content"].strip()

    # 5. Collect source metadata
    sources = [{"title": h["title"], "score": h["score"]} for h in hits]

    return {"question": question, "answer": answer, "sources": sources}


# --- Quick smoke test ---
result = rag_answer("What is the flat fare from JFK to Manhattan?")
print(f"Q: {result['question']}")
print(f"A: {result['answer']}")
print(f"Sources: {result['sources']}")
print("\n\u2713 R-05 complete: rag_answer() validated")

## R-06 — Test with sample questions

In [0]:
# R-06 — Test RAG with sample questions & document answer quality

sample_questions = [
    "What is the standard taxi rate in NYC?",
    "When is taxi demand highest in Manhattan?",
    "How is the MTA tax applied to taxi fares?",
    "What payment types do NYC taxis accept?",
    "What is the JFK flat rate?",
    "Are drivers required to use the taximeter?",
    "What surcharges apply at night?",
    "Can I hail a green taxi in Times Square?",
]

quality_log = []

for idx, q in enumerate(sample_questions, 1):
    print(f"\n{'=' * 70}")
    print(f"Q{idx}: {q}")
    print("=" * 70)

    result = rag_answer(q, top_k=2)

    # Display answer and sources
    print(f"\nAnswer:\n  {result['answer']}")
    print("\nSources:")
    for src in result["sources"]:
        print(f"  - {src['title']}  (score: {src['score']:.4f})")

    # --- Automated quality assessment ---
    top_score = result["sources"][0]["score"] if result["sources"] else 0.0
    answer_lower = result["answer"].lower()

    # Relevance: top retrieval score
    if top_score >= 0.72:
        relevance = "High"
    elif top_score >= 0.60:
        relevance = "Medium"
    else:
        relevance = "Low"

    # Groundedness: does the answer avoid hedging / "not enough info"?
    hedges = ["not enough information", "context does not", "i don't have"]
    grounded = "No" if any(h in answer_lower for h in hedges) else "Yes"

    # Conciseness: roughly 3-5 sentences expected
    sentence_count = len([s for s in result["answer"].split(".") if s.strip()])
    concise = "Yes" if 2 <= sentence_count <= 7 else "No"

    quality_log.append(
        {
            "question": q,
            "relevance": relevance,
            "grounded": grounded,
            "concise": concise,
            "top_score": round(top_score, 4),
            "answer_preview": result["answer"][:120],
        }
    )

    print(
        f"\n  Quality  \u2192  relevance={relevance}  grounded={grounded}  concise={concise}"
    )

# --- Summary table ---
print(f"\n\n{'=' * 70}")
print("R-06 QUALITY SUMMARY")
print("=" * 70)
print(
    f"{'#':<4} {'Relevance':<11} {'Grounded':<10} {'Concise':<9}"
    f" {'Top Score':<10} {'Question'}"
)
print("-" * 90)
for i, entry in enumerate(quality_log, 1):
    print(
        f"{i:<4} {entry['relevance']:<11} {entry['grounded']:<10}"
        f" {entry['concise']:<9} {entry['top_score']:<10}"
        f" {entry['question'][:45]}"
    )

high_count = sum(1 for e in quality_log if e["relevance"] == "High")
grounded_count = sum(1 for e in quality_log if e["grounded"] == "Yes")
print(
    f"\n\u2713 R-06 complete: {len(sample_questions)} questions tested"
    f" | {high_count}/{len(quality_log)} high-relevance"
    f" | {grounded_count}/{len(quality_log)} grounded"
)

## Answer quality log

| # | Question | Relevance | Grounded | Concise | Top Score | Notes |
|---|----------|-----------|----------|---------|-----------|-------|
| 1 | What is the standard taxi rate in NYC? | High | Yes | No | 0.7475 | Correct fare breakdown; slightly verbose |
| 2 | When is taxi demand highest in Manhattan? | Medium | Yes | Yes | 0.6881 | Correctly states corpus lacks demand data |
| 3 | How is the MTA tax applied to taxi fares? | High | Yes | Yes | 0.7725 | Accurate $0.50 surcharge; well-cited |
| 4 | What payment types do NYC taxis accept? | High | Yes | Yes | 0.7901 | Lists cash, card, contactless correctly |
| 5 | What is the JFK flat rate? | Medium | Yes | Yes | 0.6237 | Correct $70 answer; lower retrieval score |
| 6 | Are drivers required to use the taximeter? | High | Yes | Yes | 0.7555 | Accurately cites TLC rules |
| 7 | What surcharges apply at night? | Medium | Yes | Yes | 0.6367 | Correct $1.00 overnight surcharge |
| 8 | Can I hail a green taxi in Times Square? | High | Yes | Yes | 0.7971 | Correctly infers Times Square is below 110th St |

**Summary:** 5/8 high-relevance, 8/8 grounded, 7/8 concise. The pipeline correctly refuses to hallucinate on Q2 (demand data not in corpus) and makes a sound spatial inference on Q8.

## BQ-5 — Interactive Q&A Interface

Type a question in the **widget above** and run the cell below to get an answer grounded in the TLC policy corpus.

---

In [0]:
# BQ-5 — Interactive conversational interface for non-technical stakeholders
dbutils.widgets.text("question", "What is the JFK flat rate?", "Ask a question")

user_question = dbutils.widgets.get("question").strip()

if not user_question:
    print("⚠ Please enter a question in the widget above and re-run this cell.")
else:
    result = rag_answer(user_question, top_k=2)

    # ── Formatted output ──────────────────────────────────────────────────
    print("═" * 70)
    print(f"  ❓  {result['question']}")
    print("═" * 70)
    print(f"\n  {result['answer']}\n")
    print("─" * 70)
    print("  Sources:")
    for src in result["sources"]:
        print(f"    • {src['title']}  (relevance: {src['score']:.4f})")
    print("─" * 70)